# 01_深度研究 (Deep Research Agent)

**来源:** [LangChain Docs — Build a deep research agent](https://docs.langchain.com/deep-agents/deep-research)

本教程演示如何从头开始构建一个多步骤的 Web 研究智能体。该智能体将研究问题分解为聚焦的子任务，委派给专门的子智能体，并将发现综合成一份带有引用的综合报告。

## 1. 核心概念

| 概念 | 说明 |
|------|------|
| 子智能体 (Subagents) | 用于并行、上下文隔离的研究任务 |
| 自定义工具 | 使用 Tavily 进行 Web 搜索 |
| 多步规划 | 内置 `planning` 工具用于任务分解 |
| 编排工作流 | 主智能体协调多个子智能体并综合结果 |

## 2. 环境准备

安装依赖并设置 API 密钥。

In [ ]:
# 安装核心依赖
# pip install deepagents tavily-python httpx markdownify langchain-anthropic langchain-core

# 设置 API 密钥（按需选择模型提供商）
import os
from dotenv import load_dotenv
load_dotenv()

# os.environ["ANTHROPIC_API_KEY"] = "your-api-key"
# os.environ["TAVILY_API_KEY"] = "your-api-key"

print("环境准备完成")


## 3. 定义搜索工具

使用 Tavily 搜索 API 获取网页链接，然后抓取完整网页内容供智能体分析。

In [ ]:
# 导入必要的库
import os
from typing import Annotated, Literal
import httpx
from langchain.tools import InjectedToolArg, tool
from markdownify import markdownify
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

def fetch_webpage_content(url, timeout=10.0):
    """获取网页内容并转换为 Markdown 格式"""
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        resp = httpx.get(url, headers=headers, timeout=timeout)
        resp.raise_for_status()
        return markdownify(resp.text)
    except Exception as e:
        return f"获取失败: {e}"

@tool(parse_docstring=True)
def tavily_search(query, max_results=1, topic="general"):
    """搜索 Web 获取指定查询的信息。
    使用 Tavily 发现相关 URL，然后获取并返回完整的网页内容。
    参数:
        query: 要执行的搜索查询
        max_results: 返回的最大结果数（默认: 1）
    返回:
        带有完整网页内容的格式化搜索结果
    """
    results = tavily_client.search(query, max_results=max_results, topic=topic)
    texts = []
    for r in results.get("results", []):
        content = fetch_webpage_content(r["url"])
        texts.append(f"## {r[chr(34)+chr(116)+chr(105)+chr(116)+chr(108)+chr(101)+chr(34)]}\n**URL:** {r[chr(34)+chr(117)+chr(114)+chr(108)+chr(34)]}\n\n{content}\n---")
    return "找到 " + str(len(texts)) + " 条结果:\n\n" + "\n".join(texts)

print("搜索工具定义完成")


## 4. 定义提示模板

研究智能体需要三个核心提示模板：研究工作流指令、研究员指令、子智能体委派指令。

In [ ]:
# 编排工作流指令：定义主智能体的研究流程
RESEARCH_WORKFLOW_INSTRUCTIONS = """# 研究工作流

收到研究请求后，请按以下流程执行：

1. **规划**：使用 write_todos 创建任务清单，将研究分解为聚焦的子任务
2. **保存请求**：将用户的研究问题保存到 /research_request.md
3. **研究**：使用 task() 工具将研究任务委派给子智能体
4. **综合**：审阅所有子智能体的发现，合并引用来源
5. **撰写报告**：将最终综合报告写入 /final_report.md
6. **验证**：确认已覆盖所有方面

## 报告撰写指南
- 使用清晰的章节标题
- 用段落形式撰写
- 使用 [1][2][3] 格式内联引用来源
- 末尾添加 ### 来源 章节
"""

# 研究员指令：定义子智能体的研究行为
RESEARCHER_INSTRUCTIONS = """你是一名研究助理，负责对用户的输入主题进行研究。

你的工作是使用工具收集与用户输入主题相关的信息。
你可以使用 tavily_search 工具查找有助于回答研究问题的资源。

请遵循以下步骤：
1. **仔细阅读问题**
2. **从较宽泛的搜索开始**
3. **每次搜索后暂停并评估**
4. **随着信息收集执行更精确的搜索**
5. **当可以自信回答时停止**

工具调用预算：
- 简单查询：最多 2-3 次搜索
- 复杂查询：最多 5 次搜索
"""

# 子智能体委派指令：定义何时以及如何并行委派任务
SUBAGENT_DELEGATION_INSTRUCTIONS = """# 子智能体研究协调

你的职责是通过从待办事项列表中委派任务给子智能体来协调研究。

## 委派策略
- 默认：大多数查询从 1 个子智能体开始
- 仅在查询需要对比或有独立方面时才并行化

## 关键原则
- 偏向单子智能体
- 避免过早分解
- 仅在有明确对比需求时并行化

## 并行执行限制
- 每次迭代最多使用 {max_concurrent_research_units} 个并行子智能体
"""

print("提示模板定义完成")


## 5. 创建智能体

整合所有组件并创建深度研究智能体。

In [ ]:
from datetime import datetime
from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model

max_concurrent_research_units = 3
max_researcher_iterations = 3
current_date = datetime.now().strftime('%Y-%m-%d')

INSTRUCTIONS = (
    RESEARCH_WORKFLOW_INSTRUCTIONS
    + '\n\n' + '=' * 80 + '\n\n'
    + SUBAGENT_DELEGATION_INSTRUCTIONS.format(
        max_concurrent_research_units=max_concurrent_research_units,
    )
)

research_sub_agent = {
    "name": "research-agent",
    "description": "将研究任务委派给子智能体",
    "system_prompt": RESEARCHER_INSTRUCTIONS,
    "tools": [tavily_search],
}
model = init_chat_model(model="deepseek-v4-flash", temperature=0.0)

agent = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt=INSTRUCTIONS,
    subagents=[research_sub_agent],
)
print("深度研究智能体创建完成")


## 6. 运行研究任务

向智能体提交一个研究查询，观察它如何规划、委派并综合结果。

In [ ]:
# 示例：运行深度研究任务
# 注意：实际运行时需要有效的 API 密钥
# result = agent.invoke({
#     "messages": [{"role": "user", "content": "2025年AI发展趋势"}]
# })
# print(result["messages"][-1].content)
print("运行研究任务（API 密钥有效时取消注释）")


## 7. 关键设计原则

| 原则 | 说明 |
|------|------|
| 偏向单子智能体 | 一个综合研究任务比多个窄任务更节省 Token |
| 避免过早分解 | 不要将「研究 X」拆分为多个子任务 |
| 仅在明确比较时并行 | 对比不同实体时才使用多子智能体 |
| 每次搜索后评估 | 检查已找到什么、还缺什么 |
| 工具调用预算 | 简单查询最多 2-3 次，复杂最多 5 次 |

---

**参考链接**
- [LangChain Docs](https://docs.langchain.com/deep-agents)
- [Tavily Search API](https://tavily.com)
